In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [2]:
# objects.json is always next to this notebook
data_path = Path("objects.json")
with data_path.open("r", encoding="utf-8") as f:
  raw = json.load(f)

In [3]:
df = pd.DataFrame.from_dict(raw, orient="index").reset_index(drop=True)

In [4]:
df.shape

(3971, 52)

In [5]:
df.columns

Index(['objectID', 'isHighlight', 'accessionNumber', 'accessionYear',
       'isPublicDomain', 'primaryImage', 'primaryImageSmall',
       'additionalImages', 'constituents', 'department', 'objectName', 'title',
       'culture', 'period', 'objectDate', 'objectBeginDate', 'objectEndDate',
       'medium', 'dimensions', 'measurements', 'creditLine', 'classification',
       'metadataDate', 'repository', 'objectURL', 'tags', 'isTimelineWork',
       'objectWikidata_URL', 'GalleryNumber', 'artistRole',
       'artistDisplayName', 'artistDisplayBio', 'artistAlphaSort',
       'artistNationality', 'artistBeginDate', 'artistEndDate',
       'artistWikidata_URL', 'artistULAN_URL', 'artistPrefix', 'country',
       'geographyType', 'state', 'region', 'city', 'river', 'artistSuffix',
       'dynasty', 'reign', 'locale', 'locus', 'subregion', 'excavation'],
      dtype='object')

In [6]:
df.head()


,objectID,isHighlight,accessionNumber,accessionYear,isPublicDomain,primaryImage,primaryImageSmall,additionalImages,constituents,department,...,region,city,river,artistSuffix,dynasty,reign,locale,locus,subregion,excavation
0,44793,False,50.61.11,1950,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,44817,False,1975.268.185,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[https://images.metmuseum.org/CRDImages/as/ori...,None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,44830,False,1975.268.378,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,44862,False,1975.268.473,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[https://images.metmuseum.org/CRDImages/as/ori...,None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,44907,False,1975.268.376,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map
Possible columns:
- department
- culture
- country
- state
- region
- city
- geographyType

In [7]:
cols_to_check = [
    "department",
    "culture",
    "country",
    "state",
    "region",
    "city",
    "geographyType",
    "artistNationality",
]

results = []

for col in cols_to_check:
    s = df[col]

    # Start with nulls
    missing = s.isna()

    # For string-like values, also treat blank/whitespace/"undefined" as missing
    s_str = s.astype("string").str.strip()
    missing = missing | s_str.eq("") | s_str.str.lower().eq("undefined")

    present = ~missing

    total = len(s)
    present_count = int(present.sum())
    missing_count = int(missing.sum())

    results.append({
        "column": col,
        # "present_count": present_count,
        # "missing_count": missing_count,
        "present_pct": round(present_count / total * 100, 2),
        # "missing_pct": round(missing_count / total * 100, 2),
    })

summary = pd.DataFrame(results).sort_values("present_pct", ascending=False)
summary

,column,present_pct
0,department,100.00
1,culture,71.37
2,country,25.59
6,geographyType,12.31
7,artistNationality,11.41
5,city,6.50
4,region,4.41
3,state,3.88


In [8]:
# Count of rows per department
department_counts = (
    df["department"]
    .fillna("<<MISSING_DEPARTMENT>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_DEPARTMENT>>")
    .value_counts(dropna=False)
)

# department_counts

department_summary = pd.DataFrame({
    "count": department_counts,
    "percent": (department_counts / department_counts.sum() * 100).round(2)
})

department_summary

,count,percent
department,,
Asian Art,1981,49.89
European Sculpture and Decorative Arts,765,19.26
The Michael C. Rockefeller Wing,537,13.52
Islamic Art,337,8.49
Medieval Art,128,3.22
Robert Lehman Collection,77,1.94
Greek and Roman Art,70,1.76
The Cloisters,34,0.86
Egyptian Art,27,0.68


In [9]:
# For each department, compute % present for every column in cols_to_check

def is_present(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    return series.notna() & s.ne("") & s.str.lower().ne("undefined")

# Clean department labels for grouping
dept_series = (
    df["department"]
    .fillna("<<MISSING_DEPARTMENT>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_DEPARTMENT>>")
)

# Build a table: rows = department, cols = indicators for presence
out = pd.DataFrame({"department": dept_series})

for col in cols_to_check:
    out[f"{col}_present"] = is_present(df[col]).astype(int)

dept_present_pct = (
    out.groupby("department")[[f"{c}_present" for c in cols_to_check]]
    .mean()
    .mul(100)
    .round(0)
)

# rename "country_present" -> "country", etc.
dept_present_pct.columns = [c.replace("_present", "") for c in dept_present_pct.columns]

# add total rows per department
dept_counts = out.groupby("department").size().rename("total_count")
dept_present_pct = dept_present_pct.join(dept_counts)

# optional: show count first
dept_present_pct = dept_present_pct[["total_count"] + cols_to_check]

dept_present_pct

,total_count,department,culture,country,state,region,city,geographyType,artistNationality
department,,,,,,,,,
Ancient West Asian Art,5,100.0,80.0,0.0,0.0,80.0,0.0,0.0,0.0
Asian Art,1981,100.0,100.0,0.0,0.0,0.0,0.0,0.0,2.0
Egyptian Art,27,100.0,0.0,74.0,0.0,70.0,0.0,100.0,0.0
European Sculpture and Decorative Arts,765,100.0,0.0,0.0,0.0,0.0,0.0,0.0,53.0
Greek and Roman Art,70,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
Islamic Art,337,100.0,0.0,100.0,0.0,0.0,59.0,100.0,0.0
Medieval Art,128,100.0,100.0,73.0,63.0,0.0,21.0,74.0,0.0
Modern and Contemporary Art,10,100.0,40.0,0.0,0.0,0.0,0.0,0.0,100.0
Robert Lehman Collection,77,100.0,99.0,0.0,0.0,0.0,0.0,0.0,1.0


Findings:
- 100% have "department"
- 71.10% have "culture"
- 25.12% have "country"


Location by department:
- Asian Art: culture
- European Sculpture and Decorative Arts: artistNationality
- Islamic Art: city, country
- Medieval Art: city, state, country, culture
- Modern and Contemporary Art: culture, artistNationality
- Robert Lehman Collection: culture, artistNationality
- The Cloisters: city, state, country, culture
- The Michael C. Rockefeller Wing: city, state, country, culture

In [10]:
# Make a copy
df_loc = df.copy()

def clean_missing(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    return s.mask(s.eq("") | s.str.lower().eq("undefined"))

# Department -> ordered source columns for location
location_priority = {
    "Asian Art": ["culture"],
    "European Sculpture and Decorative Arts": ["artistNationality"],
    "Islamic Art": ["city", "country"],
    "Medieval Art": ["city", "state", "country", "culture"],
    "Modern and Contemporary Art": ["culture", "artistNationality"],
    "Robert Lehman Collection": ["culture", "artistNationality"],
    "The Cloisters": ["city", "state", "country", "culture"],
    "The Michael C. Rockefeller Wing": ["city", "state", "country", "culture"],
    "Egyptian Art": ["country", "region"],
    "Greek and Roman Art": ["culture"]
}

# Start empty
df_loc["location"] = pd.NA

# Fill location by department using first non-missing column in priority order
for dept, cols in location_priority.items():
    mask = df_loc["department"].eq(dept)
    if not mask.any():
        continue

    # build first-non-missing across preferred columns
    candidates = pd.DataFrame({c: clean_missing(df_loc.loc[mask, c]) for c in cols})
    df_loc.loc[mask, "location"] = candidates.bfill(axis=1).iloc[:, 0]

# quick check
# df_loc[["department", "culture", "artistNationality", "city", "state", "country", "location"]].head(20)

In [11]:
# How many got location per listed department
check = (
    df_loc[df_loc["department"].isin(location_priority)]
    .assign(location_present=lambda d: d["location"].notna())
    .groupby("department")["location_present"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)
check

department
Asian Art                                 100.00
European Sculpture and Decorative Arts    100.00
Greek and Roman Art                       100.00
Medieval Art                              100.00
Modern and Contemporary Art               100.00
Robert Lehman Collection                  100.00
The Cloisters                             100.00
The Michael C. Rockefeller Wing           100.00
Islamic Art                                99.70
Egyptian Art                               77.78
Name: location_present, dtype: float64

In [12]:
location_counts = (
    df_loc["location"]
    .fillna("<<MISSING_LOCATION>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_LOCATION>>")
    .value_counts(dropna=False)
)

location_summary = pd.DataFrame({
    # "count": location_counts,
    "percent": (location_counts / location_counts.sum() * 100).round(2)
})

location_summary

,percent
location,
China,31.65
Japan,14.51
Peru,10.85
British,8.21
French,4.86
...,...
Kufa,0.03
Saintonge,0.03
probably Basra,0.03


In [13]:
n_unique_locations = (
    df_loc["location"]
    .fillna("<<MISSING_LOCATION>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_LOCATION>>")
    .nunique(dropna=False)
)

print(f"Number of different locations: {n_unique_locations:,}")

Number of different locations: 175


In [14]:
all_locations = (
    df_loc["location"]
    .fillna("<<MISSING_LOCATION>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_LOCATION>>")
    .drop_duplicates()
    .sort_values()
    .tolist()
)

all_locations

['<<MISSING_LOCATION>>',
 'Abu Mena',
 'Alta Verapaz',
 'American',
 'American and French',
 'Ancash',
 'Arizona',
 'Belgian',
 'Bohemian',
 'Bolivia',
 'British',
 'British, Scottish',
 'Byzantine (Egypt)',
 'Chiclayo',
 'Chihuahua',
 'China',
 'China (?)',
 'China or Japan (?)',
 'ChinaNeihuLu',
 'Chinese',
 'Cocle Province',
 'Colima',
 'Colombia',
 'Comala',
 'Ctesiphon',
 'Cypriot',
 'Danish',
 'Danish (Nästved)',
 'Derbyshire',
 'Dutch',
 'Dutch (Delft)',
 'Eastern Syria',
 'Ecuador',
 'Egypt',
 'Etruscan',
 'Europe',
 'European',
 'Faliscan',
 'Florence',
 'Florence or its vicinity',
 'Fomena (?)',
 'French',
 'French (Limoges)',
 'French (Paris)',
 'Fustat',
 'German',
 'Greek',
 'Greek, Attic',
 'Greek, Corinthian',
 'Greek, Egypt, Alexandria-Hadra',
 'Greek, South Italian, Apulian',
 'Greek, South Italian, Apulian, Canosan',
 'Greek, South Italian, Campanian',
 'Greek, probably Cypriot',
 'Guatemala',
 'Guatemala or Mexico',
 'Gurgan',
 'Homs',
 'India',
 'India, Bombay Schoo

### Timeline
- objectBeginDate
- objectEndDate
- objectDate

In [15]:
date_cols = ["objectBeginDate", "objectEndDate", "objectDate"]

rows = []
for col in date_cols:
    s = df[col]
    s_str = s.astype("string").str.strip()
    present = s.notna() & s_str.ne("") & s_str.str.lower().ne("undefined")
    rows.append({
        "column": col,
        "present_count": int(present.sum()),
        "missing_count": int((~present).sum()),
        "present_pct": round(present.mean() * 100, 2),
    })

pd.DataFrame(rows)

,column,present_count,missing_count,present_pct
0,objectBeginDate,3971,0,100.00
1,objectEndDate,3971,0,100.00
2,objectDate,3608,363,90.86


In [16]:
b = pd.to_numeric(df["objectBeginDate"], errors="coerce")
e = pd.to_numeric(df["objectEndDate"], errors="coerce")

both_present = b.notna() & e.notna()
equal = both_present & (b == e)

n_total = len(df)
n_both = int(both_present.sum())
n_equal = int(equal.sum())

print(f"Rows where begin & end are both present:  ({n_both / n_total * 100:.2f}%)")
print(f"Rows where objectBeginDate == objectEndDate (and both present): ({n_equal / n_total * 100:.2f}%)")
# print(f"Among rows with both present: {n_equal / n_both * 100:.2f}%" if n_both else "Among rows with both present: n/a (no rows)")

Rows where begin & end are both present:  (100.00%)
Rows where objectBeginDate == objectEndDate (and both present): (6.42%)


In [17]:
df[date_cols].sample(n=min(20, len(df)), random_state=42)

,objectBeginDate,objectEndDate,objectDate
1860,1200,1299,13th century
1419,1600,1633,early 17th century
2687,1800,1899,19th century
211,1667,1722,late 17th–early 18th century
1490,1630,1630,1630
298,1662,1722,NaN
2801,1740,1755,ca. 1745–50
1861,-500,-100,5th–2nd century BCE
3970,-175,325,200 BCE–300 CE
2281,575,750,late 6th–mid-8th century


In [18]:
df2 = df.copy()
b = pd.to_numeric(df2["objectBeginDate"], errors="coerce")
e = pd.to_numeric(df2["objectEndDate"], errors="coerce")
mid = ((b + e) / 2).where(b.notna() & e.notna())
df2["final_date"] = (mid // 1).where(mid.notna()).astype("Int64")

In [19]:
df2['final_date'].sample(n=min(20, len(df)), random_state=42)

1860    1249
1419    1616
2687    1849
211     1694
1490    1630
298     1692
2801    1747
1861    -300
3970      75
2281     662
2342    1880
2107    1849
2456    1749
3615    1692
2742    1824
3476       7
149     1695
3395     -49
1791    1747
2862    1849
Name: final_date, dtype: Int64

In [20]:
df2["final_date"].agg(["min", "max"])

min   -3515
max    1950
Name: final_date, dtype: int64